# STR-3 — Backpressure & micro-batch sizing

**Break → Detect → Fix → Prove.** A Structured Streaming query runs as a series of
**micro-batches**. How many records each batch pulls is the biggest lever on a streaming job's
behavior: pull **too many** and a batch goes huge → memory pressure, long batch times, and a
flood of writes (the streaming **small-files** problem); pull **too few** and per-batch overhead
dominates. The Kafka source option **`maxOffsetsPerTrigger`** (file source: `maxFilesPerTrigger`)
**caps the input per micro-batch**.

**Pre-requisite:** the unified Spark server is up (`make up`). Producers use the host listener
`localhost:29092`; Spark's `readStream` uses the internal listener `kafka:9092` (`SPARK_BOOTSTRAP`).
**Open the Spark UI at http://localhost:4040 → Structured Streaming tab** and watch it while the
cells run.

**Laptop-safe:** a bounded ~5,000-event batch; every query uses `.trigger(availableNow=True)` so it
**drains what's there and stops on its own** (no infinite stream). Checkpoints/sinks live under
`.tmp/`; the topic + sinks are removed at teardown.

See the companion writeup in [`README.md`](./README.md) and the
[troubleshooting sheet](../../docs/troubleshooting.md). Small-files ties to **LAK-2**.

In [ ]:
from common.spark_session import spark, display_df
from common.kafka_helpers import ensure_topic, produce_events, delete_topic, SPARK_BOOTSTRAP
from common.iceberg_meta import table_health
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, LongType, StringType

TOPIC      = "str3_orders"
N_EVENTS   = 5000           # bounded backlog (laptop-safe)
SINK_BIG   = "iceberg_catalog.default.str3_sink_unbounded"   # 'default' namespace is pre-created
SINK_SMALL = "iceberg_catalog.default.str3_sink_bounded"
CKPT_BIG   = "./.tmp/checkpoint_str3_unbounded"
CKPT_SMALL = "./.tmp/checkpoint_str3_bounded"

print("Spark reads via", SPARK_BOOTSTRAP, "| Spark UI -> Structured Streaming:", "http://localhost:4040")
spark

## Step 0 — a topic + a bounded backlog

Create a single-partition topic and burst ~5,000 events onto it. Each event carries a small text
`payload` so the data has realistic **bytes** (not just rows) — that's what makes a real backlog's
single batch expensive, and what the Iceberg sink actually writes. Nothing streams yet; this is the
backlog a catch-up run will have to drain.

In [ ]:
ensure_topic(TOPIC, num_partitions=1, recreate=True)

# value_fn pads each event with ~120 bytes so batches carry real payload, not just a counter.
produce_events(
    TOPIC, N_EVENTS,
    value_fn=lambda i: {"order_id": i, "amount": (i % 1000) * 1.5, "payload": f"order-{i}-" + ("x" * 100)},
)
print(f"produced {N_EVENTS:,} events to {TOPIC} (the backlog to drain)")

## Helpers — a JSON schema and a reader we can re-parametrize

We parse each Kafka value as JSON, then write the flattened rows into Iceberg. The only thing that
changes between the broken and fixed runs is **one reader option** (`maxOffsetsPerTrigger`), so we
wrap the read+write in a small helper. `availableNow=True` makes each query drain all available data
and **stop**, then `awaitTermination()` blocks until it's done.

In [ ]:
schema = StructType([
    StructField("order_id", LongType(),   True),
    StructField("amount",   StringType(), True),   # JSON number -> read as string, kept simple
    StructField("payload",  StringType(), True),
])

def run_stream(sink_table, checkpoint, max_offsets=None):
    """Drain the topic into `sink_table` with availableNow; cap per-batch input if max_offsets set.
    Returns the (terminated) streaming query so you can read q.recentProgress."""
    reader = (spark.readStream.format("kafka")
              .option("kafka.bootstrap.servers", SPARK_BOOTSTRAP)
              .option("subscribe", TOPIC)
              .option("startingOffsets", "earliest"))
    if max_offsets is not None:
        reader = reader.option("maxOffsetsPerTrigger", max_offsets)   # <- the per-batch cap

    rows = (reader.load()
            .select(F.from_json(F.col("value").cast("string"), schema).alias("d"))
            .select("d.order_id", "d.amount", "d.payload"))

    q = (rows.writeStream.format("iceberg").outputMode("append")
         .option("checkpointLocation", checkpoint)
         .trigger(availableNow=True)
         .toTable(sink_table))
    q.awaitTermination()        # bounded: returns when the backlog is fully drained
    return q

def batch_shape(q):
    """[(batchId, numInputRows), ...] from a finished query's recentProgress."""
    return [(p["batchId"], int(p["numInputRows"])) for p in q.recentProgress if p.get("numInputRows", 0)]

import shutil
def reset(sink_table, checkpoint):
    """Clean slate so batch shape & data-file counts are deterministic across re-runs."""
    spark.sql(f"DROP TABLE IF EXISTS {sink_table}")
    shutil.rmtree(checkpoint, ignore_errors=True)

## 1. Break it — no cap → one giant batch

Read the whole backlog with `availableNow` and **no** `maxOffsetsPerTrigger`. Spark pulls everything
available in a **single micro-batch**. At 5,000 rows that's harmless, but it *is* the failure mode in
miniature: a real backlog of millions would be one massive in-memory batch (memory pressure, long
batch time, a write that's hard to size).

In [ ]:
reset(SINK_BIG, CKPT_BIG)
q_big = run_stream(SINK_BIG, CKPT_BIG, max_offsets=None)
shape_big = batch_shape(q_big)
print("batch shape (batchId, numInputRows):", shape_big)
print(f"-> {len(shape_big)} batch(es); total rows = {sum(n for _, n in shape_big):,}")

## 2. Detect it — read the batch shape & the sink's files

Two views of the same thing:
- **`q.recentProgress`** — the per-batch `numInputRows` series. Unbounded = **one** entry holding
  *all* the rows. (Spark UI → **Structured Streaming** tab shows the same as a single fat bar.)
- **`table_health(spark, SINK)`** — the Iceberg sink's **data-file** count: writes-per-batch
  materialized as files. One batch → few files now, but a real one-shot batch can still emit many
  parts and pin memory while it does.

In [ ]:
h_big = table_health(spark, SINK_BIG, "unbounded (1 giant batch)")
print(f"batches      : {len(shape_big)}")
print(f"rows/batch   : {[n for _, n in shape_big]}")
print(f"data files   : {h_big['data_files']}   (avg file size {h_big['avg_file_bytes']:,} B)")

## 3. Fix it — bound the batch with `maxOffsetsPerTrigger`

Set `.option("maxOffsetsPerTrigger", 1000)`. Now even under `availableNow`, Spark drains the **same**
~5,000-record backlog across **~5 bounded micro-batches** of ~1,000 rows each — steady, predictable
memory and latency no matter how big the backlog is. (The file-source equivalent is
`maxFilesPerTrigger`.)

In [ ]:
reset(SINK_SMALL, CKPT_SMALL)
q_small = run_stream(SINK_SMALL, CKPT_SMALL, max_offsets=1000)
shape_small = batch_shape(q_small)
print("batch shape (batchId, numInputRows):", shape_small)
print(f"-> {len(shape_small)} batches; total rows = {sum(n for _, n in shape_small):,}")

## 4. Prove it

Same total rows, different batch shape. The batch count going **1 → ~5** while the total stays
~5,000 is the proof the cap converted one unbounded gulp into bounded, steady batches. The sink
**data-file** counts show the **small-files tie** (LAK-2): more frequent bounded batches write more
(smaller) files — which is why you schedule `rewrite_data_files` / `OPTIMIZE` alongside a streaming
sink.

In [ ]:
h_small = table_health(spark, SINK_SMALL, "bounded (maxOffsetsPerTrigger=1000)")

def summarize(label, shape, h):
    rows = sum(n for _, n in shape)
    print(f"{label:<34} batches={len(shape):<3} total_rows={rows:<6,} "
          f"data_files={h['data_files']:<3} avg_file={h['avg_file_bytes']:,}B")

print("maxOffsetsPerTrigger:  size batches to your memory + latency budget\n")
summarize("broken  (no cap)",         shape_big,   h_big)
summarize("fixed   (cap = 1000)",     shape_small, h_small)
print("\nSame total rows; bounded run trades 1 huge batch for ~5 steady ones (more, smaller files -> compact).")

## Takeaways & "in real production…"

- **Bound your batches.** Set `maxOffsetsPerTrigger` (Kafka) / `maxFilesPerTrigger` (files) so peak
  memory and batch latency track your **budget**, not the size of an unexpected backlog.
- **Size it to memory + latency.** Bigger batches = fewer files + more throughput but higher peak
  memory/latency; smaller / more frequent = lower latency but more overhead and **more small files**.
- **Mind the small-files tie (LAK-2).** Frequent bounded triggers keep memory safe but multiply tiny
  files — schedule compaction (`rewrite_data_files` / `OPTIMIZE`) alongside a streaming sink.
- **Watch the Structured Streaming tab.** Per-batch input rows + batch duration are the live signals;
  a single fat batch (or batch time climbing toward the trigger interval) means you're unbounded or
  under-provisioned.

## Teardown

Delete the topic and drop both Iceberg sinks. `make clean` also clears the checkpoints and any local
`.tmp/` state.

In [ ]:
delete_topic(TOPIC)
for t in (SINK_BIG, SINK_SMALL):
    spark.sql(f"DROP TABLE IF EXISTS {t}")
print("Deleted", TOPIC, "and dropped both sinks. (`make clean` clears checkpoints + .tmp.)")